# Blender Visualisation — FOGGIE Density Grid Export

Extracts 3D Cartesian density grids from a raw FOGGIE snapshot using `yt`,
then saves them as numpy arrays ready for conversion to OpenVDB and import into Blender.

**Pipeline:**
```
Raw snapshot  →  yt arbitrary_grid  →  .npy files  →  convert_to_vdb.py  →  Blender
```

**Run this notebook on the server** (foggie conda env), then transfer the `.npy` files
to your local machine for VDB conversion and Blender import.

## 0. Configuration

In [ ]:
# ── What to export ────────────────────────────────────────────────────────────
HALO_ID   = "halo_008508"   # e.g. "halo_008508", "halo_005036"
REDSHIFT  = 0.0             # redshift to visualise

# Grid resolution: 128 = fast preview, 256 = good quality, 512 = final render
RESOLUTION = 128

# Box half-width as a fraction of R200 (1.2 = slightly beyond the virial radius)
EXTENT_R200 = 1.2

# Which components to export
COMPONENTS = ["gas", "dm", "star"]

# Log-normalise arrays to [0,1] before saving?
# Recommended: True — makes Blender shader tuning much easier
LOG_NORMALISE = True

## 1. Imports and paths

In [ ]:
import os
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import yt

yt.set_log_level("warning")
%matplotlib inline

# ── project paths (same as your analysis notebooks) ──────────────────────────
os.chdir("/home/shanson/foggie-dynamics")
sys.path.insert(0, ".")

from src import ProjectPaths, load_hdf5
from src.filters_and_utils import filter_highres_particles

PATHS = ProjectPaths.create(
    code_root=Path("/home/shanson/foggie-dynamics"),
    data_root=Path("/cephfs/shanson/FOGGIE"),
)

print(f"Data root : {PATHS.data_root}")
print(f"Results   : {PATHS.results}")

## 2. Load metadata (centre and R200)

Your existing pipeline already computed the halo centre and virial radius
and stored them in `metadata.h5` — we just read those directly.

In [ ]:
data_dir = PATHS.snapshot_data(HALO_ID, REDSHIFT)
meta = load_hdf5(data_dir / "metadata.h5")

centre_kpc = np.array(meta["center_kpc"])   # [x, y, z] in kpc
R200_kpc   = float(meta["R200_kpc"])
extent_kpc = EXTENT_R200 * R200_kpc

print(f"Halo    : {HALO_ID}  z={REDSHIFT}")
print(f"Centre  : {centre_kpc} kpc")
print(f"R200    : {R200_kpc:.1f} kpc")
print(f"Box     : ±{extent_kpc:.1f} kpc  ({EXTENT_R200}×R200)")
print(f"Grid    : {RESOLUTION}³ = {RESOLUTION**3:,} voxels")

## 3. Load raw snapshot with yt

In [ ]:
# ── Register particle filters (same as your analysis code) ────────────────────
def _star_filter(pfilter, data):
    return data["all", "creation_time"] > 0

def _dm_filter(pfilter, data):
    return data["all", "creation_time"] <= 0

yt.add_particle_filter("star", function=_star_filter, filtered_type="all", requires=["creation_time"])
yt.add_particle_filter("dm",   function=_dm_filter,   filtered_type="all", requires=["creation_time"])

# ── Find and load snapshot ────────────────────────────────────────────────────
# Raw snapshots live at: data_root / halo_id / nref11c_nref9f / RD... / RD...
#from src.workflow import find_snapshots

halo_raw_path = Path(f"/cephfs/brs/FOGGIE/{HALO_ID}/nref11c_nref9f")
#snaps = find_snapshots(halo_raw_path, snapshot_type="RD")

snap_path = halo_raw_path / "RD0042/RD0042"
print(f"Loading snapshot: {snap_path}")

ds = yt.load(str(snap_path))
ds.add_particle_filter("star")
ds.add_particle_filter("dm")

print(f"Loaded  : z = {ds.current_redshift:.4f}  "
      f"box = {ds.domain_width.in_units('Mpc')[0]:.1f} Mpc")

## 4. Set up extraction box

In [ ]:
# Convert centre and extent to yt quantities in the dataset's units
centre_yt = ds.arr(centre_kpc, "kpc")
extent_yt = ds.quan(extent_kpc, "kpc")

left_edge  = centre_yt - extent_yt
right_edge = centre_yt + extent_yt

dims = [RESOLUTION, RESOLUTION, RESOLUTION]

print(f"Extraction box:")
print(f"  Left  : {left_edge.in_units('kpc')}")
print(f"  Right : {right_edge.in_units('kpc')}")
print(f"  Dims  : {dims}")

## 5. Extract density grids

`ds.arbitrary_grid` resamples the AMR data onto a uniform Cartesian mesh.
For particle components (DM, stars) it uses CIC deposition.

In [ ]:
def log_normalise(arr, floor_percentile=2.0):
    nonzero = arr[arr > 0]
    if len(nonzero) == 0:
        return arr.astype(np.float32)
    floor  = np.percentile(nonzero, floor_percentile)
    safe   = np.where(arr > floor, arr / floor, 1.0)
    result = np.where(arr > floor, np.log10(safe), 0.0)
    if result.max() > 0:
        result /= result.max()
    return result.clip(0, 1).astype(np.float32)


KPC_TO_CM = 3.0857e21   # conversion factor used throughout


def particles_to_grid(ds, centre_yt, extent_kpc, resolution, particle_type):
    """Deposit particles onto a 3D grid via histogramdd (mass-weighted)."""
    sphere = ds.sphere(centre_yt, ds.quan(extent_kpc * 1.05, "kpc"))
    pos    = sphere[particle_type, "particle_position"].in_units("kpc").value
    mass   = sphere[particle_type, "particle_mass"].in_units("g").value

    c      = centre_yt.in_units("kpc").value
    pos_c  = pos - c
    mask   = np.all(np.abs(pos_c) < extent_kpc, axis=1)
    pos_c, mass = pos_c[mask], mass[mask]
    print(f"  {particle_type}: {mask.sum():,} particles in box")
    if mask.sum() == 0:
        return np.zeros((resolution,) * 3, dtype=np.float32)

    edges   = [np.linspace(-extent_kpc, extent_kpc, resolution + 1)] * 3
    grid, _ = np.histogramdd(pos_c, bins=edges, weights=mass)
    vox_cm3 = ((2 * extent_kpc / resolution) * KPC_TO_CM) ** 3
    density = grid / vox_cm3
    print(f"  density: min={density.min():.2e}  max={density.max():.2e} g/cm³")
    return density.astype(np.float32)


def gas_to_grid(ds, centre_yt, extent_kpc, resolution):
    """
    Deposit AMR gas cells onto a 3D grid via histogramdd.
    Uses ds.sphere() directly — avoids arbitrary/covering grid issues entirely.
    Each cell contributes mass = density * cell_volume to its voxel.
    """
    sphere = ds.sphere(centre_yt, ds.quan(extent_kpc * 1.05, "kpc"))

    x    = sphere["gas", "x"].in_units("kpc").value
    y    = sphere["gas", "y"].in_units("kpc").value
    z    = sphere["gas", "z"].in_units("kpc").value
    rho  = sphere["gas", "density"].in_units("g/cm**3").value
    vol  = sphere["gas", "cell_volume"].in_units("kpc**3").value

    c    = centre_yt.in_units("kpc").value
    xc, yc, zc = x - c[0], y - c[1], z - c[2]
    mask = (np.abs(xc) < extent_kpc) & (np.abs(yc) < extent_kpc) & (np.abs(zc) < extent_kpc)

    pos  = np.column_stack([xc[mask], yc[mask], zc[mask]])
    mass = rho[mask] * vol[mask] * KPC_TO_CM**3   # density [g/cm³] * volume [cm³] → mass [g]

    print(f"  gas: {mask.sum():,} cells in box")
    print(f"  gas density in cells: min={rho[mask].min():.2e}  max={rho[mask].max():.2e} g/cm³")

    edges   = [np.linspace(-extent_kpc, extent_kpc, resolution + 1)] * 3
    grid, _ = np.histogramdd(pos, bins=edges, weights=mass)
    vox_cm3 = ((2 * extent_kpc / resolution) * KPC_TO_CM) ** 3
    density = grid / vox_cm3

    print(f"  grid: min={density.min():.2e}  max={density.max():.2e} g/cm³")
    return density.astype(np.float32)


# ── Extract all three components ──────────────────────────────────────────────
grids = {}

print("Extracting gas (sphere → histogramdd)...")
gas = gas_to_grid(ds, centre_yt, extent_kpc, RESOLUTION)
gas = np.nan_to_num(gas, nan=0.0).clip(0)
grids["gas"] = log_normalise(gas) if LOG_NORMALISE else gas

print("\nExtracting dark matter...")
dm = particles_to_grid(ds, centre_yt, extent_kpc, RESOLUTION, "dm")
dm = np.nan_to_num(dm, nan=0.0).clip(0)
grids["dm"] = log_normalise(dm) if LOG_NORMALISE else dm

print("\nExtracting stars...")
star = particles_to_grid(ds, centre_yt, extent_kpc, RESOLUTION, "star")
star = np.nan_to_num(star, nan=0.0).clip(0)
grids["star"] = log_normalise(star) if LOG_NORMALISE else star

print("\nDone. Components extracted:", list(grids.keys()))

In [ ]:
# ── Diagnostic: where is the data in each grid? ───────────────────────────
for comp, grid in grids.items():
    print(f"\n{comp}:")
    print(f"  global max = {grid.max():.4f}  nonzero voxels = {np.count_nonzero(grid):,}")
    
    # Check max value in each z-slice — tells us where data is concentrated
    z_maxes = [grid[:, :, z].max() for z in range(RESOLUTION)]
    peak_z = int(np.argmax(z_maxes))
    print(f"  peak z-slice index: {peak_z}  (midplane = {RESOLUTION//2})")
    print(f"  top 5 z-slices by max: {sorted(range(RESOLUTION), key=lambda z: z_maxes[z])[-5:]}")

## 6. Quick sanity check — midplane slice plots

Before saving, check the grids look right visually.

In [ ]:
colours = {"gas": "Blues", "dm": "Purples", "star": "Oranges"}
mid = RESOLUTION // 2

n_comp = len(grids)
fig, axes = plt.subplots(1, n_comp, figsize=(5 * n_comp, 5))
if n_comp == 1:
    axes = [axes]

for ax, (comp, grid) in zip(axes, grids.items()):
    slice_2d = grid[:, :, mid]  # z-midplane slice
    im = ax.imshow(
        slice_2d.T,
        origin="lower",
        cmap=colours.get(comp, "viridis"),
        vmin=0, vmax=1 if LOG_NORMALISE else None,
        extent=[-extent_kpc, extent_kpc, -extent_kpc, extent_kpc],
    )
    ax.set_title(comp, fontsize=13)
    ax.set_xlabel("x [kpc]")
    ax.set_ylabel("y [kpc]")

    # Overlay R200 circle
    circle = plt.Circle((0, 0), R200_kpc, color="white", fill=False,
                         linestyle="--", linewidth=1, alpha=0.6)
    ax.add_patch(circle)
    plt.colorbar(im, ax=ax, label="norm. density")

fig.suptitle(f"{HALO_ID}  z={REDSHIFT}  —  midplane slice  (z = 0 kpc)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("If slices look centred and show clear structure, grids are good to export.")

## 7. Save numpy arrays

In [ ]:
out_dir = PATHS.results / HALO_ID / "blender"
out_dir.mkdir(parents=True, exist_ok=True)

for comp, grid in grids.items():
    out_path = out_dir / f"{comp}_density.npy"
    np.save(str(out_path), grid)
    size_mb = out_path.stat().st_size / 1e6
    print(f"  {comp:4s} → {out_path.name}  shape={grid.shape}  {size_mb:.1f} MB")

# Save metadata alongside the arrays
metadata = {
    "halo":         HALO_ID,
    "redshift":     float(ds.current_redshift),
    "R200_kpc":     R200_kpc,
    "extent_kpc":   extent_kpc,
    "resolution":   RESOLUTION,
    "log_norm":     LOG_NORMALISE,
    "centre_kpc":   centre_kpc.tolist(),
    "components":   list(grids.keys()),
}
with open(out_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Saved to: {out_dir}")
print(f"\nNext steps:")
print(f"  1. Transfer {out_dir} to your local machine")
print(f"  2. conda install -c conda-forge pyopenvdb")
print(f"  3. python code/convert_to_vdb.py --halo 8508")
print(f"  4. Open Blender → run code/blender_scene.py")

## 8. Optional — projection plots (like yt slice plots)

Max-intensity projections along each axis — gives a feel for the 3D structure.

In [ ]:
comp_to_plot = "gas"   # change to "dm" or "star"
grid = grids[comp_to_plot]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axis_labels = [("y", "z", 0), ("x", "z", 1), ("x", "y", 2)]

for ax, (xlabel, ylabel, axis) in zip(axes, axis_labels):
    proj = grid.max(axis=axis)   # max-intensity projection
    ax.imshow(
        proj.T, origin="lower",
        cmap=colours.get(comp_to_plot, "viridis"),
        extent=[-extent_kpc, extent_kpc, -extent_kpc, extent_kpc],
    )
    ax.set_xlabel(f"{xlabel} [kpc]")
    ax.set_ylabel(f"{ylabel} [kpc]")
    circle = plt.Circle((0, 0), R200_kpc, color="white", fill=False,
                         linestyle="--", linewidth=1, alpha=0.5)
    ax.add_patch(circle)

fig.suptitle(f"{comp_to_plot} — max-intensity projections", fontsize=13)
plt.tight_layout()
plt.show()